In [5]:
# XML Parsing

import xml.etree.ElementTree as ET

ssml_data = ""

with open("sentence_01.xml","r", encoding="utf-8") as file:
    ssml_data = file.read()

root = ET.fromstring(ssml_data)

print(f"Root tag: {root.tag}") 

for paragraph in root.findall('p'):
    for sentence in paragraph.findall('s'):
        print(f"Found sentence text: {''.join(sentence.itertext())}")


Root tag: speak
Found sentence text: यो कौण है?
Found sentence text: यो कौण है?


In [ ]:
# Structure Analysis

import nltk

nltk.download('punkt_tab') 
from nltk.tokenize import sent_tokenize
with open("raw_text.txt","r", encoding="utf-8") as file:
    raw_text = file.read() 
sentences = sent_tokenize(raw_text)

for index, sentence in enumerate(sentences):
    print(f"Sentence {index + 1}: {sentence}")

In [ ]:
# Structure Analysis

import re

with open("raw_text.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

sentences = re.split(r'[।?!]+', raw_text)

sentences = [s.strip() for s in sentences if s.strip()]

for index, sentence in enumerate(sentences):
    print(f"Sentence {index + 1}: {sentence}")

In [ ]:
import re

# Hindi number-to-words mapping
_HI_ONES = ["", "एक", "दो", "तीन", "चार", "पाँच", "छह", "सात", "आठ", "नौ"]
_HI_TEENS = ["दस", "ग्यारह", "बारह", "तेरह", "चौदह", "पंद्रह", "सोलह", "सत्रह", "अठारह", "उन्नीस"]
_HI_TENS = ["", "", "बीस", "तीस", "चालीस", "पचास", "साठ", "सत्तर", "अस्सी", "नब्बे"]

def num_to_hindi(n):
    """Convert an integer to Hindi words."""
    if n == 0:
        return "शून्य"
    parts = []
    if n >= 10000000:
        parts.append(num_to_hindi(n // 10000000) + " करोड़")
        n %= 10000000
    if n >= 100000:
        parts.append(num_to_hindi(n // 100000) + " लाख")
        n %= 100000
    if n >= 1000:
        parts.append(num_to_hindi(n // 1000) + " हज़ार")
        n %= 1000
    if n >= 100:
        parts.append(_HI_ONES[n // 100] + " सौ")
        n %= 100
    if n >= 20:
        parts.append(_HI_TENS[n // 10])
        n %= 10
    elif n >= 10:
        parts.append(_HI_TEENS[n - 10])
        n = 0
    if 0 < n < 10:
        parts.append(_HI_ONES[n])
    return " ".join(parts)

def process_sub_tags(xml_tree):
    for sub_tag in xml_tree.findall('.//sub'):
        alias_text = sub_tag.get('alias')
        if alias_text:
            sub_tag.text = alias_text
    return xml_tree

def automated_text_normalization(raw_text):
    
    def currency_repl(match):
        amount = int(match.group(1))
        words = num_to_hindi(amount)
        return f"{words} रूपये"
    
    text = re.sub(r'₹(\d+)', currency_repl, raw_text)
    
    def number_repl(match):
        number = int(match.group(0))
        return num_to_hindi(number)
    
    text = re.sub(r'\b\d+\b', number_repl, text)
    
    return text

ssml_text = "मेरे धोरे ₹200 है।"

normalized_text = automated_text_normalization(ssml_text)
print(f"Normalized Text: {normalized_text}")

In [ ]:
import re

# Hindi number-to-words mapping
_HI_ONES = ["", "एक", "दो", "तीन", "चार", "पाँच", "छह", "सात", "आठ", "नौ"]
_HI_TEENS = ["दस", "ग्यारह", "बारह", "तेरह", "चौदह", "पंद्रह", "सोलह", "सत्रह", "अठारह", "उन्नीस"]
_HI_TENS = ["", "", "बीस", "तीस", "चालीस", "पचास", "साठ", "सत्तर", "अस्सी", "नब्बे"]

def num_to_hindi(n):
    """Convert an integer to Hindi words."""
    if n == 0:
        return "शून्य"
    parts = []
    if n >= 10000000:
        parts.append(num_to_hindi(n // 10000000) + " करोड़")
        n %= 10000000
    if n >= 100000:
        parts.append(num_to_hindi(n // 100000) + " लाख")
        n %= 100000
    if n >= 1000:
        parts.append(num_to_hindi(n // 1000) + " हज़ार")
        n %= 1000
    if n >= 100:
        parts.append(_HI_ONES[n // 100] + " सौ")
        n %= 100
    if n >= 20:
        parts.append(_HI_TENS[n // 10])
        n %= 10
    elif n >= 10:
        parts.append(_HI_TEENS[n - 10])
        n = 0
    if 0 < n < 10:
        parts.append(_HI_ONES[n])
    return " ".join(parts)

def automated_text_normalization(raw_text):

    def currency_repl(match):
        amount = int(match.group(1))
        words = num_to_hindi(amount)
        return f"{words} रुपये"

    # Convert ₹200
    text = re.sub(r'₹(\d+)', currency_repl, raw_text)

    # Convert remaining numbers
    def number_repl(match):
        number = int(match.group(0))
        return num_to_hindi(number)

    text = re.sub(r'\b\d+\b', number_repl, text)

    return text


ssml_text = "मेरे धोरे ₹200 है।"

normalized_text = automated_text_normalization(ssml_text)

print(f"Normalized Text: {normalized_text}")

In [ ]:
import xml.etree.ElementTree as ET
from g2p_en import G2p

# Initialize the G2P engine (which contains both a dictionary and fallback rules)
g2p = G2p()

def text_to_phonemes(xml_string):
    root = ET.fromstring(xml_string)
    phoneme_output = []
    
    # Iterate through the elements
    for element in root.iter():
        # Path 1: Explicit <phoneme> markup
        if element.tag == 'phoneme':
            # Grab the exact phonemes provided by the author
            explicit_ph = element.attrib.get('ph')
            if explicit_ph:
                phoneme_output.append(f"[{explicit_ph}]")
                
        # Path 2: Automated fallback for standard text
        # (Assuming the text is normalized and not inside a phoneme tag)
        elif element.text and element.text.strip() and element.tag != 'phoneme':
            words = element.text.strip().split()
            for word in words:
                # Ask the G2P engine to guess the phonemes
                phonemes = g2p(word)
                # g2p returns a list of phonemes, let's join them
                phoneme_output.append(" ".join(phonemes))
                
    return " | ".join(phoneme_output)

# --- Pipeline Execution ---
# Notice we have a normal word and a hard-to-pronounce name marked up explicitly
ssml_data = """<?xml version="1.0"?>
<speak>
  Hello 
  <phoneme alphabet="ipa" ph="ˈwʊk.i">Wookiee</phoneme>
</speak>
"""

phoneme_sequence = text_to_phonemes(ssml_data)
print(f"Phoneme Sequence: {phoneme_sequence}")

# OUTPUT:
# Phoneme Sequence: HH AH0 L OW1 | [ˈwʊk.i]

In [ ]:
from google.cloud import texttospeech

# 1. Read your XML file
with open("sentence_01.xml", "r", encoding="utf-8") as file:
    ssml_text = file.read()

client = texttospeech.TextToSpeechClient()
synthesis_input = texttospeech.SynthesisInput(ssml=ssml_text)
voice = texttospeech.VoiceSelectionParams(language_code="hi-IN", name="hi-IN-Neural2-A")
audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)

# 2. Send to the API
response = client.synthesize_speech(input=synthesis_input, voice=voice, audio_config=audio_config)

# 3. Save the resulting audio
with open("output.mp3", "wb") as out:
    out.write(response.audio_content)

In [ ]:
def ssml_to_audio(ssml_text: str) -> None:
    """
    Generates audio from SSML text using Google Cloud Text-to-Speech API.

    Args:
        ssml_text: string of SSML text
    """

    # Instantiates a client
    client = texttospeech.TextToSpeechClient()

    # Sets the text input to be synthesized
    synthesis_input = texttospeech.SynthesisInput(ssml=ssml_text)

    # Builds the voice request
    voice = texttospeech.VoiceSelectionParams(
        language_code="hi-IN", name="hi-IN-Neural2-A"
    )

    # Selects the type of audio file to return
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    # Performs the text-to-speech request
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    # Writes the synthetic audio to the output file.
    with open("test_example.mp3", "wb") as out:
        out.write(response.audio_content)
        print("Audio content written to file test_example.mp3")

In [ ]:
import asyncio
import edge_tts

with open("sentence_01.xml", "r", encoding="utf-8") as file:
    text = file.read()

async def generate_audio():
    tts = edge_tts.Communicate(text, voice="hi-IN-MadhurNeural")
    await tts.save("test.mp3")

await generate_audio()